In [ ]:
from pathlib import Path


TRAIN_CONFIG_PATH = Path(
    "/ceph/mri.meduniwien.ac.at/departments/"
    "radiology/mrsbrain/public/hfish/walinet/"
    "configs/Training/train_7T.yaml"
)

SEED = 123456

N_EXAMPLES = 100_000
BATCH_SIZE = 4096

In [ ]:
from pathlib import Path
import sys


project_root = Path.cwd().resolve()

while not (
    project_root
    / "src"
    / "walinet"
).is_dir():
    if project_root.parent == project_root:
        raise FileNotFoundError(
            "WALINET-Projektordner nicht gefunden."
        )

    project_root = project_root.parent


src_dir = project_root / "src"

if str(src_dir) not in sys.path:
    sys.path.insert(
        0,
        str(src_dir),
    )


print(
    "WALINET source:",
    src_dir,
)


from walinet.visualization.VisualizeSim import (
    plot_spectra_range,
)

In [ ]:
import torch

from walinet.training_data.build_simulation_system import (
    build_simulation_system,
)


system = build_simulation_system(
    TRAIN_CONFIG_PATH
)

train_cfg = system.train_config
simulation_cfg = system.simulation_config
resources = system.resources
pool = resources.train

prepared_basis = system.prepared_basis
metabolite_simulator = system.metabolite_simulator
spectrum_simulator = system.train_simulator
device = system.device


generator = torch.Generator(
    device=device
)

generator.manual_seed(
    SEED
)


print(
    "Simulation system ready."
)

print(
    "Device:",
    device,
)

print(
    "Simulation config:",
    system.simulation_config_path,
)

print(
    "Basis library:",
    simulation_cfg.basis.library,
)

print(
    "Training subjects:",
    pool.subject_names,
)

In [ ]:
batch = spectrum_simulator.simulate(
    batch_size=N_EXAMPLES,
    generator=generator,
)

total_spectra = batch.normalized_input_spectra
baseline_spectra = batch.normalized_target_spectra

metabolite_spectra = (
    total_spectra
    - baseline_spectra
)

In [ ]:
plot_spectra_range(
    total_spectra=total_spectra,
    metabolite_spectra=metabolite_spectra,
    simulation_cfg=simulation_cfg,
    start=20,
    stop=30,
    component="real",
    ppm_min=0.0,
    ppm_max=7.0,
)

In [ ]:
batch = spectrum_simulator.simulate(
    batch_size=128,
    generator=generator,
)

print(batch.network_input.shape)
print(batch.network_target.shape)
print(
    None
    if batch.network_l2 is None
    else batch.network_l2.shape
)